# 03 Preprocessing

Stage 3 turns the register's human text into a rule a program can evaluate against a date and a
time, and knows when it cannot. This notebook is the team's walkthrough of that stage: what each
raw field says, how it is parsed, every spelling the city uses and what we made of it, and how
each area ends up with a status that says how far its answer can be trusted.

**The principle.** The app never answers more confidently than the data supports. For a parser
that means **failing closed**: it must account for every character of a field, and anything it
cannot fully read becomes an issue with a reason, never a partial answer.

**How it is built.** The parsing is [`pipeline/rules.py`](../../pipeline/rules.py) and its
application is [`pipeline/process.py`](../../pipeline/process.py), the exact code that produced the
committed data and the app's file. This notebook imports them and runs them on the committed
snapshot, which keeps the register's raw fields beside the parsed ones, so every step can be
shown and traced back. It runs offline.

**Run it** with the "Python (parkkiko)" kernel, then *Restart & Run All*. Section 7 reads an
earlier version of the parser from git history, so it needs a full clone.

**Contents**
1. What goes in
2. Rule type: the class and the space type together
3. Hours
4. Time limits
5. Seasons
6. Free-text notes
7. Failing closed, and what it changed
8. From fields to a status
9. Which window applies on a given day
10. Checks before anything is written
11. What comes out
12. Does the committed data match the code?
13. What this stage hands on
14. Reflection on the stage

In [1]:
import importlib.util
import json
import os
import subprocess
import sys
import tempfile
from datetime import date
from pathlib import Path

# Run from the repository root, so paths mean what they mean in pipeline/
os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline").is_dir()))
sys.path[:0] = ["pipeline", "analysis"]

import geopandas as gpd
import pandas as pd
from IPython.display import Markdown

import export_web
import process
import report_stats as rs
import rules

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

areas = gpd.read_parquet(rs.SOURCE)
RAW = ["luokka", "luokka_nimi", "tyyppi", "voimassaolo", "kesto", "kausi", "lisatieto"]
print(f"{len(areas):,} areas; raw fields kept alongside the parsed ones: {', '.join(RAW)}")

8,754 areas; raw fields kept alongside the parsed ones: luokka, luokka_nimi, tyyppi, voimassaolo, kesto, kausi, lisatieto


## 1. What goes in

Seven fields carry the rule, all written by people for people (their meanings are in
[02 data collection](02_data_collection.ipynb), section 3). The number of different ways each
field is written is the size of the problem:

In [2]:
Markdown(rs.raw_spellings(areas))

### Distinct raw spellings per field

| Field                        |   Distinct spellings |   Missing or blank |
|:-----------------------------|---------------------:|-------------------:|
| Hours (`voimassaolo`)        |                   66 |               2922 |
| Time limit (`kesto`)         |                   25 |               6247 |
| Season (`kausi`)             |                   15 |               8274 |
| Space type (`tyyppi`)        |                   25 |               6866 |
| Free-text note (`lisatieto`) |                  201 |               6993 |


Sixty-six ways to write opening hours, 25 to write a time limit. The approach for every field is
the same: list every distinct spelling, decide what each one means, and write a parser that
accounts for all of them, so a spelling it has never seen is flagged rather than guessed.

## 2. Rule type: the class and the space type together

The first question for any area is what kind of rule it has. The register answers it in two
fields: the parking class `luokka`, a number with a Finnish name, and the space type `tyyppi`,
which marks special bays. Every class has one meaning, kept as data in `rules.LUOKKA_RULES`:

In [3]:
pd.DataFrame(
    [(code, rule, limit, name) for code, (rule, limit, name) in rules.LUOKKA_RULES.items()],
    columns=["class", "rule type", "limit stated by the class, min", "meaning"],
).set_index("class")

,rule type,"limit stated by the class, min",meaning
class,,,
1,free_limited,NaN,Free short-term parking
2,free_limited,NaN,Free long-term parking
3,paid,60.0,"Single payment, up to 1 hour"
4,paid,120.0,"Single payment, up to 2 hours"
5,paid,240.0,"Single payment, up to 4 hours"
6,paid,NaN,Paid without a resident or business permit; hours vary by location
7,paid,60.0,Single payment up to 1 hour without a resident or business permit
8,free_limited,NaN,Free short-term parking without a permit; use a parking disc
9,banned_hours,NaN,Parking allowed outside the no-parking hours


The class alone is not enough. Class 0, or no class at all, is where the city puts both parking
bans and reserved bays, and only the space type tells them apart. A space type also overrides an
ordinary class: a taxi bay in a paid street is not paid parking. The table below shows how each
class splits once the space type is taken into account:

In [4]:
klass = areas["luokka"].fillna(0).astype(int).rename("class")
pd.crosstab(klass, areas["rule_type"], margins=True, margins_name="all")

rule_type,always_banned,banned_hours,free_limited,paid,reserved,unknown,all
class,,,,,,,
0,884,0,0,0,980,62,1926
1,0,0,406,0,0,0,406
2,0,0,31,0,3,0,34
3,0,0,0,30,0,0,30
4,0,0,0,83,0,0,83
5,0,0,0,522,0,0,522
6,0,0,0,3089,0,0,3089
7,0,0,0,429,0,0,429
8,1,0,1598,0,0,0,1599


`rules.rule_type` applies that logic. An unrecognised space type is an issue, not a fall-through
to the class: the city adds new space types, and falling back would turn a new kind of reserved
bay into "paid" with full confidence.

In [5]:
examples = [(6, None), (6, "Taxi"), (0, "Pysäköintikielto"), (None, "Sähköpotkulauta"),
            (6, "Uusi merkki")]
pd.DataFrame([(k, t, *rules.rule_type(k, t)) for k, t in examples],
             columns=["class", "space type", "rule type", "limit, min", "issue"])

,class,space type,rule type,"limit, min",issue
0,6.0,NaN,paid,None,None
1,6.0,Taxi,reserved,None,None
2,0.0,Pysäköintikielto,always_banned,None,None
3,NaN,Sähköpotkulauta,reserved,None,None
4,6.0,Uusi merkki,unknown,None,"(unreadable, unknown space type 'Uusi merkki')"


## 3. Hours

`voimassaolo` states when the rule is in force, in the convention of Finnish street signs: a bare
range is Monday to Friday, a range in brackets is Saturday, and a third range is Sunday. So
`9-21, (9-18)` means weekdays 9 to 21 and Saturday 9 to 18. A sign with no Saturday range says
nothing about Saturday, which means the rule is not in force then.

`rules.parse_hours` turns that into one range per day type, or an issue:

In [6]:
for raw in ["9-21", "9-21, (9-18)", "8-18, (8-14), 10-14", "9.21, (9-18)", "7-9, 15-17", "8-18 la"]:
    print(f"{raw!r:24} -> {rules.parse_hours(raw)}")

'9-21'                   -> ({'mon_fri': [9, 21]}, None)
'9-21, (9-18)'           -> ({'mon_fri': [9, 21], 'sat': [9, 18]}, None)
'8-18, (8-14), 10-14'    -> ({'mon_fri': [8, 18], 'sat': [8, 14], 'sun': [10, 14]}, None)
'9.21, (9-18)'           -> ({'mon_fri': [9, 21], 'sat': [9, 18]}, None)
'7-9, 15-17'             -> (None, ('ambiguous', "two ranges with no brackets to tell the days apart: '7-9, 15-17'"))
'8-18 la'                -> (None, ('unreadable', "the hours '8-18 la' also say 'la', which we cannot read"))


`9.21, (9-18)` is a real typo in the register, a full stop for a dash, and is accepted on
purpose. `7-9, 15-17` is the one genuinely ambiguous value in the data: two ranges with no
brackets could be weekdays and Sunday or a split day, so it is flagged. `8-18 la` is invented, to
show what happens to a word the parser does not know. Here is **every** spelling in the register
and what it became:

In [7]:
values = areas["voimassaolo"].dropna()
print(f"{(values.str.strip() == '').sum() + areas['voimassaolo'].isna().sum():,} areas leave it blank")
spellings = values[values.str.strip() != ""].value_counts()
parsed = [rules.parse_hours(raw) for raw in spellings.index]
pd.DataFrame({
    "areas": spellings.values,
    "parsed": [value for value, issue in parsed],
    "issue": [issue[0] if issue else "" for value, issue in parsed],
}, index=pd.Index(spellings.index, name="raw"))

2,922 areas leave it blank


,areas,parsed,issue
raw,,,
"9-21, (9-18)",1907,"{'mon_fri': [9, 21], 'sat': [9, 18]}",
9-21,1708,"{'mon_fri': [9, 21]}",
8-20,1487,"{'mon_fri': [8, 20]}",
8-17,151,"{'mon_fri': [8, 17]}",
"9-21, (9-21), 9-21",114,"{'mon_fri': [9, 21], 'sat': [9, 21], 'sun': [9, 21]}",
8-18,92,"{'mon_fri': [8, 18]}",
"7-18, (9-15)",36,"{'mon_fri': [7, 18], 'sat': [9, 15]}",
7-18,30,"{'mon_fri': [7, 18]}",
9-21\n,30,"{'mon_fri': [9, 21]}",


## 4. Time limits

`kesto` is the longest a car may stay. `rules.parse_duration` returns minutes: `0` means the
register explicitly says there is no limit (`ei aikarajaa`), and nothing means none was stated,
which is not the same thing. Some values repeat themselves in Russian (`30 min, (30 мин)`), and
the parser accepts exactly that duplicate and nothing else after the comma.

In [8]:
values = areas["kesto"].dropna()
print(f"{(values.str.strip() == '').sum() + areas['kesto'].isna().sum():,} areas leave it blank")
spellings = values[values.str.strip() != ""].value_counts()
parsed = [rules.parse_duration(raw) for raw in spellings.index]
pd.DataFrame({
    "areas": spellings.values,
    "parsed": [value for value, issue in parsed],
    "issue": [issue[0] if issue else "" for value, issue in parsed],
}, index=pd.Index(spellings.index, name="raw"))

6,247 areas leave it blank


,areas,parsed,issue
raw,,,
4 h,1794,240.0,
60 min,473,60.0,
2 h,105,120.0,
24 h,22,1440.0,
4h,16,240.0,
30 min,15,30.0,
"30 min, (30 мин)",14,30.0,
"60 min, (60 мин)",12,60.0,
max 60 min lauantai,8,NaN,unreadable


`max 60 min lauantai`, "maximum 60 minutes on Saturday", hides a condition about the day inside a
field meant for a duration. Reading it as 60 minutes would drop the Saturday, so it is flagged.

A class can state a limit too (class 5 is "single payment, up to 4 hours"). Where the class and
`kesto` disagree, `process.classify` flags the contradiction instead of choosing one:

In [9]:
contradictions = areas["reason"].str.contains("the class allows", na=False)
areas.loc[contradictions, ["id", "luokka", "kesto", "reason"]]

,id,luokka,kesto,reason
1326,9385,5.0,24 h,the class allows 240 min but the register says 1440 min
1427,9487,5.0,24 h,the class allows 240 min but the register says 1440 min
1565,9626,3.0,30min,the class allows 60 min but the register says 30 min
1590,9651,5.0,24 h,the class allows 240 min but the register says 1440 min
1744,9810,5.0,2 h,the class allows 240 min but the register says 120 min
2141,10211,5.0,24 h,the class allows 240 min but the register says 1440 min
3268,11361,5.0,24 h,the class allows 240 min but the register says 1440 min
4306,12428,5.0,24 h,the class allows 240 min but the register says 1440 min
8286,16582,4.0,4,the class allows 120 min but the register says 240 min
8287,16583,4.0,4,the class allows 120 min but the register says 240 min


## 5. Seasons

`kausi` restricts a rule to part of the year. `0` and `ympärivuotinen` mean all year, and so does
a range from 1 January to 31 December however it is written.

In [10]:
values = areas["kausi"].dropna()
print(f"{(values.str.strip() == '').sum() + areas['kausi'].isna().sum():,} areas leave it blank")
spellings = values[values.str.strip() != ""].value_counts()
parsed = [rules.parse_season(raw) for raw in spellings.index]
pd.DataFrame({
    "areas": spellings.values,
    "parsed": [value for value, issue in parsed],
    "issue": [issue[0] if issue else "" for value, issue in parsed],
}, index=pd.Index(spellings.index, name="raw"))

8,274 areas leave it blank


,areas,parsed,issue
raw,,,
0,410,None,
(1.1.-31.12),27,None,
(1.1.-31.12.),11,None,
1.4. - 31.10.,10,"{'start': [4, 1], 'end': [10, 31]}",
(1.4.-31.10.),7,"{'start': [4, 1], 'end': [10, 31]}",
Ympärivuotinen,4,None,
(1.5. - 31.10.),2,"{'start': [5, 1], 'end': [10, 31]}",
Vain sataman luvalla,2,None,unreadable
(1.11.-31.3.),1,"{'start': [11, 1], 'end': [3, 31]}",


## 6. Free-text notes

`lisatieto` is free text. Most of it is a sign reference number, which says nothing about the
rule and is kept as metadata. The rest is a condition written in prose that we deliberately do
not parse: it is shown to the driver verbatim, and the area is marked uncertain.

In [11]:
notes = areas["lisatieto"].dropna()
notes = notes[notes.str.strip().ne("") & notes.ne("0")]
kinds = notes.map(lambda raw: "condition we do not parse" if rules.parse_extra(raw)[1] else "sign reference")
print(kinds.value_counts().to_string(), "\n")
notes[kinds == "condition we do not parse"].value_counts().head(10)

lisatieto
sign reference               1061
condition we do not parse     301 



lisatieto
Moottorikäyttöisillä ajoneuvoilla ajo kielletty 22-6 (22-6) 22-6, Ei koske huoltoajoa, tontilleajoa, asukaspysäköintitunnuksella F, Gäller ej servicekörning, tomtinfart, med boendeparkeringstecknet F    66
C41, E8                                                                                                                                                                                                    41
reunakiven päällä, 6909                                                                                                                                                                                    29
7539, C39                                                                                                                                                                                                  16
C41                                                                                                                                                                   

Night-time driving bans, permit exceptions, private-property notices: each could change the
answer, and none can be read reliably with patterns. Reading them is the stretch goal for a
language model (board task S1), and until then the honest answer is to show the text.

But not everything in that list is a condition. `C41, E8` and `7539, C39` look like traffic sign
codes: the letter is the sign's type in the Finnish system. The parser recognises only digits as a
sign reference, so a code with a letter counts as a condition:

In [12]:
sign_code = r"(?:[A-Z]\d{1,3}(?:\.\d)?|\d+)"
only_codes = (notes.str.fullmatch(rf"\s*{sign_code}(?:\s*[,/]\s*{sign_code})*[\s,/]*")
              & (kinds != "sign reference"))
flagged_for_it = areas["lisatieto"].isin(notes[only_codes]) & (areas["issue_codes"] == "note")
print(f"notes that are only sign codes with a letter: {only_codes.sum()}")
print(f"areas uncertain for that reason alone: {flagged_for_it.sum()}")
areas.loc[flagged_for_it, "rule_type"].value_counts()

notes that are only sign codes with a letter: 116
areas uncertain for that reason alone: 116


rule_type
reserved         89
always_banned    27
Name: count, dtype: int64

All of them are reserved bays or no-parking areas, whose rule does not depend on a sign reference,
yet each tells the driver *check the sign*. That errs in the safe direction: it costs an answer,
never a fine. It is probably too safe, and changing it changes the app's data, so it is raised as
an open question on the board rather than changed here.

## 7. Failing closed, and what it changed

Every parser returns `(value, issue)` and must account for every character of its field. Our
first parser, committed on 20 September, did not: it read what it recognised and dropped the
rest. We can measure what that meant by loading it from git history and running both versions
over every spelling in the register:

In [13]:
def parser_at(commit):
    """pipeline/rules.py as it was at a commit, loaded as a module."""
    source = subprocess.run(["git", "show", f"{commit}:pipeline/rules.py"],
                            capture_output=True, text=True, check=True).stdout
    path = Path(tempfile.mkdtemp()) / f"rules_{commit}.py"
    path.write_text(source)
    spec = importlib.util.spec_from_file_location(path.stem, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

first = parser_at("4e729f3")   # "Parse the parking register into machine-readable rules"

changed = []
for field, parser in [("voimassaolo", "parse_hours"), ("kesto", "parse_duration"),
                      ("kausi", "parse_season"), ("lisatieto", "parse_extra")]:
    for raw, n in areas[field].dropna().value_counts().items():
        old_value, old_issue = getattr(first, parser)(raw)
        new_value, new_issue = getattr(rules, parser)(raw)
        if old_issue is None and new_issue is not None:
            changed.append({"field": field, "raw": raw, "areas": n,
                            "first parser read": old_value, "now": new_issue[1]})
pd.DataFrame(changed)

,field,raw,areas,first parser read,now
0,voimassaolo,"7-15, Maksullinen (9-18)",5,"{'mon_fri': [7, 15], 'sat': [9, 18]}","the hours '7-15, Maksullinen (9-18)' also say 'Maksullinen', which we cannot..."
1,voimassaolo,"7-18, Maksullinen (9-18)",3,"{'mon_fri': [7, 18], 'sat': [9, 18]}","the hours '7-18, Maksullinen (9-18)' also say 'Maksullinen', which we cannot..."


Over the register's real values, two spellings were read confidently by the first parser and are
flagged now. The first parser dropped the word *Maksullinen*, "paid", and read Saturday 9-18 as
part of the same rule. Look at the whole of those 8 areas:

In [14]:
paid_saturday = areas["voimassaolo"].str.contains("Maksullinen", na=False)
areas.loc[paid_saturday, ["id", "luokka", "rule_type", "voimassaolo", "kesto", "status"]]

,id,luokka,rule_type,voimassaolo,kesto,status
731,8778,9.0,banned_hours,"7-18, Maksullinen (9-18)",max 60 min lauantai,uncertain
873,8928,9.0,banned_hours,"7-15, Maksullinen (9-18)",max 60 min lauantai,uncertain
1152,9210,9.0,banned_hours,"7-18, Maksullinen (9-18)",max 60 min lauantai,uncertain
1233,9291,9.0,banned_hours,"7-18, Maksullinen (9-18)",max 60 min lauantai,uncertain
2282,10354,9.0,banned_hours,"7-15, Maksullinen (9-18)",max 60 min lauantai,uncertain
3712,11830,9.0,banned_hours,"7-15, Maksullinen (9-18)",max 60 min lauantai,uncertain
4921,13055,9.0,banned_hours,"7-15, Maksullinen (9-18)",max 60 min lauantai,uncertain
6615,14782,9.0,banned_hours,"7-15, Maksullinen (9-18)",max 60 min lauantai,uncertain


They are class 9, *parking allowed outside the no-parking hours*: no parking 7-18 on weekdays.
Read together, the two fields describe a **second rule** the register has no column for: on
Saturday, 9-18 is paid parking with a 60-minute limit. The first parser would have called
Saturday 9-18 a ban.

These areas were never shown to a driver as certain, but only because the *other* field,
`max 60 min lauantai`, was flagged. The right answer, by luck, resting on a neighbouring field.
Failing closed per field removes the luck: each field is now flagged on its own evidence, so
improving one parser later cannot silently expose a wrong reading in another.

The same version also read a value like `7-18 7-15` as `7-18`. That spelling is not in the
register, but it is the kind the city could add tomorrow, so it is one of the cases in
`tests/test_status.py`:

In [15]:
print("first parser:", first.parse_hours("7-18 7-15"))
print("today:       ", rules.parse_hours("7-18 7-15"))

first parser: ({'mon_fri': [7, 18]}, None)
today:        (None, ('ambiguous', "two ranges with no brackets to tell the days apart: '7-18 7-15'"))


## 8. From fields to a status

`process.classify` runs every parser on one area and combines the results into the rule, plus a
**status** that says how far the answer can be trusted:

| Status | Meaning | What the app says |
|---|---|---|
| `official` | the register states a complete rule | the answer |
| `missing_hours` | we know the kind of rule, but not when it applies | the register does not publish hours here |
| `uncertain` | something could not be read, or contradicts itself | check the sign, and why |

`missing_hours` is deliberately separate from `uncertain`: "the city does not say" is a
different, and more useful, sentence than "something is wrong here". And a ban or a reserved bay
applies at every hour, so it needs no hours and is not missing them. One area, end to end:

In [16]:
row = areas.loc[areas["id"] == 8928, RAW].iloc[0].to_dict()
print("in: ", row)
print("out:", process.classify(row))

in:  {'luokka': 9.0, 'luokka_nimi': 'Pysäköinti sallittu pysäköintikieltoajan ulkopuolella', 'tyyppi': '', 'voimassaolo': '7-15, Maksullinen (9-18)', 'kesto': 'max 60 min lauantai', 'kausi': '', 'lisatieto': ''}
out: {'rule_type': 'banned_hours', 'class_name_en': 'Parking allowed outside the no-parking hours', 'space_type_en': None, 'hours': None, 'duration_min': None, 'season': None, 'extra_info': None, 'status': 'uncertain', 'issue_codes': 'unreadable', 'reason': "the hours '7-15, Maksullinen (9-18)' also say 'Maksullinen', which we cannot read; unreadable duration 'max 60 min lauantai'"}


In [17]:
pd.crosstab(areas["rule_type"], areas["status"], margins=True, margins_name="all")

status,missing_hours,official,uncertain,all
rule_type,,,,
always_banned,0,841,44,885
banned_hours,36,234,13,283
free_limited,381,1651,3,2035
paid,561,3776,109,4446
reserved,0,889,154,1043
unknown,0,0,62,62
all,978,7391,385,8754


Every uncertain area carries an issue code, for the app to branch on, and a reason in plain
words, for the driver:

In [18]:
Markdown(rs.issue_codes(areas))

### Issue codes

| Issue code   |   Areas |
|:-------------|--------:|
| note         |     301 |
| unreadable   |      86 |
| ambiguous    |       1 |

385 uncertain areas, 3 with more than one code.


In [19]:
areas["reason"].dropna().value_counts().head(10)

reason
the sign carries a condition we do not read; check it yourself                                                                  298
no rule for class 0.0                                                                                                            48
no rule for class nan                                                                                                            11
the class allows 240 min but the register says 1440 min                                                                           6
the hours '7-15, Maksullinen (9-18)' also say 'Maksullinen', which we cannot read; unreadable duration 'max 60 min lauantai'      5
the hours '7-18, Maksullinen (9-18)' also say 'Maksullinen', which we cannot read; unreadable duration 'max 60 min lauantai'      3
the class allows 60 min but the register says 30 min                                                                              3
no rule for class 0.0; the sign carries a condition we do not read; c

## 9. Which window applies on a given day

Parsing tells us the weekday, Saturday and Sunday windows; something must decide which one applies
on a date. `rules.window_for` does, and the pipeline uses the same function to write the app's
holiday calendar: a public holiday follows the Sunday window, and the day before one follows
Saturday's. Without it, Christmas Day reads as whatever weekday it lands on.

In [20]:
import holidays
finnish = set(holidays.Finland(years=2026))
days = [date(2026, 12, 22), date(2026, 12, 23), date(2026, 12, 25), date(2026, 12, 26), date(2026, 12, 28)]
pd.DataFrame({"weekday": [d.strftime("%A") for d in days],
              "window": [rules.window_for(d, finnish) for d in days]},
             index=pd.Index(days, name="date"))

,weekday,window
date,,
2026-12-22,Tuesday,mon_fri
2026-12-23,Wednesday,sat
2026-12-25,Friday,sun
2026-12-26,Saturday,sun
2026-12-28,Monday,mon_fri


Christmas Day 2026 is a Friday, and its window is Sunday's. Which of these days the city really
treats as holidays for parking, and which we are unsure of, is the calendar described in
[02 data collection](02_data_collection.ipynb), section 7.

## 10. Checks before anything is written

`process.check` refuses to write unless ids are unique and present, every area has a geometry we
can measure, every shape is a polygon, and the coordinates are metres around Helsinki. It passes
on the committed data:

In [21]:
process.check(areas)
print("check passed")

check passed


A check you have never seen fail is a check you hope works. Two broken copies:

In [22]:
duplicated = pd.concat([areas, areas.head(1)])
in_degrees = areas.to_crs(4326).set_crs(3879, allow_override=True)   # degrees, labelled metres

for name, broken in [("a duplicated id", duplicated), ("coordinates in degrees", in_degrees)]:
    try:
        process.check(broken)
        print(f"{name}: NOT caught")
    except ValueError as error:
        print(f"{name}: refused, {str(error)[:70]}")

a duplicated id: refused, parking area ids must be unique and present
coordinates in degrees: refused, coordinates are not metres around Helsinki: [24.84436636 60.14744288 2


The second one is the coordinate trap from [02 data collection](02_data_collection.ipynb),
section 5, caught by its numbers rather than its label.

Each area is also given the district its centre lies in (`process.add_district`), by the centre
because some areas straddle a boundary and the answer must not depend on row order.

## 11. What comes out

Two files, both written from the same parsed table, so the analysis and the app cannot disagree.

**`data/processed/parking_rules.parquet`**, the analysis snapshot, committed to git. One row per
area, the register's own fields kept beside the parsed ones so any parse can be traced back:

| Register field | Becomes |
|---|---|
| `luokka`, `luokka_nimi`, `tyyppi` | `rule_type`, `class_name_en`, `space_type_en` |
| `voimassaolo` | `hours`, e.g. `{"mon_fri": [9, 21], "sat": [9, 18]}` |
| `kesto` (or the class's own limit) | `duration_min` |
| `kausi` | `season`, e.g. `{"start": [4, 1], "end": [10, 31]}` |
| `lisatieto` | `extra_info`, when it is a condition |
| all of them | `status`, `issue_codes`, `reason` |
| geometry | `district` |

The data model follows the one Mai drew up for the cleaning notebook.

In [23]:
areas.drop(columns="geometry").dtypes.rename("type").to_frame()

,type
id,int32
luokka,float64
luokka_nimi,str
tyyppi,str
voimassaolo,str
kesto,str
kausi,str
lisatieto,str
rule_type,str
class_name_en,str


**`web/public/data/parking_areas.geojson`**, what the app loads, also committed so the map runs
straight after a clone. It carries only the fields the app reads, in GPS coordinates. A field
with no value is **absent**, never null or empty: the app tests for presence, and an empty value
costs as many bytes as a real one.

In [24]:
print("fields:", export_web.WEB_FIELDS)
app = json.loads(export_web.TARGET.read_text())
for feature in app["features"]:
    if feature["properties"]["status"] == "uncertain":
        print(json.dumps(feature["properties"], ensure_ascii=False, indent=1))
        break

fields: ['id', 'rule_type', 'luokka', 'tyyppi', 'hours', 'duration_min', 'season', 'status', 'reason', 'extra_info']
{
 "id": 8015,
 "rule_type": "paid",
 "luokka": 6,
 "hours": {
  "mon_fri": [
   9,
   21
  ]
 },
 "status": "uncertain",
 "reason": "the sign carries a condition we do not read; check it yourself",
 "extra_info": "Moottorikäyttöisillä ajoneuvoilla ajo kielletty 22-6 (22-6) 22-6, Ei koske huoltoajoa, tontilleajoa, asukaspysäköintitunnuksella F, Gäller ej servicekörning, tomtinfart, med boendeparkeringstecknet F"
}


## 12. Does the committed data match the code?

Everything above demonstrates the parsers; this checks them against what is committed. Re-run
`process.classify` on the raw fields of every area and compare with the parsed columns in the
snapshot:

In [25]:
fresh = pd.DataFrame([process.classify(r) for r in areas[RAW].to_dict("records")], index=areas.index)
same = lambda a, b: (a.isna() & b.isna()) | (a.astype(object) == b.astype(object))
pd.Series({column: int((~same(fresh[column], areas[column])).sum()) for column in fresh},
          name="areas that differ").to_frame()

,areas that differ
rule_type,0
class_name_en,0
space_type_en,0
hours,0
duration_min,0
season,0
extra_info,0
status,0
issue_codes,0
reason,0


No differences: the committed snapshot is exactly what today's code produces. The app's file has
its own check in the pipeline, which rebuilds it and compares the bytes:

In [26]:
result = subprocess.run([sys.executable, "pipeline/export_web.py", "--check"], capture_output=True, text=True)
print(result.stdout or result.stderr)

web/public/data/parking_areas.geojson matches the committed data



## 13. What this stage hands on

- **To exploration (stage 4):** one row per area with a rule type, parsed hours, limit and
  season, a status, a district, and the raw fields to trace any of it back.
- **To learning (stage 5):** the 978 `missing_hours` areas to predict, and 5,800-odd areas with
  stated hours to learn from.
- **To the app:** a file whose every field is either a value the app can use or absent, and
  where every doubt arrives with a reason a driver can read.

## 14. Reflection on the stage

**What the canvas planned.** The preprocessing box set the goal of turning the register into a
structured rule the app can evaluate against a date and time: parse the hours, normalise the time
limits, combine class and space type into five rule types, match roadworks to parking areas, and
show unusual sign text to the driver as written.

**What worked.** All of it except the roadworks. The hours, limits and seasons parse for every
spelling the city uses; the class and space type combine into rule types; unusual text reaches
the driver verbatim. Section 12 shows the result reproduces exactly from the code.

**What did not, and why.** Our first parser did not fail closed. It never showed a driver a wrong
answer, but section 7 shows that was luck: on 8 areas the right outcome rested on a neighbouring
field.

**What changed from the plan, and why.**
- *A sixth rule type, `unknown`.* The canvas's five assumed every area could be classified. An
  unrecognised space type now becomes `unknown` with a reason, instead of falling back to the
  class.
- *A status for every area.* The canvas did not plan one; it became the project's central
  output, because the app's honesty depends on knowing, per area, how far the answer goes.
- *Roadworks matching dropped*, with the source itself (see 02).
- *Feature engineering moved to the learning stage*, where the canvas placed it anyway.

**What we would do differently.**
- Fail closed from the first parser, per field.
- Keep the vocabulary in one place from the start. The English labels once lived in both the
  pipeline and the cleaning notebook, and the two copies drifted apart before they were merged
  into `rules.py`.
- List every distinct value of a field *before* writing its parser, as the tables in sections
  3 to 5 now do, rather than discovering spellings from failures.

**Possible future steps.**
- Treat lettered sign codes as references (section 6), which would give 116 areas a confident
  answer, then read the remaining free-text conditions with a language model (board task S1),
  showing the reading as an estimate.
- Model more than one rule per area. The 8 Saturday areas in section 7 hold two rules in fields
  built for one, and today we can only flag them.
- Cross-check rules against Digiroad's traffic sign panels, which carry hours and limits as typed
  values.

**Experience and technical implementation.** Pure functions that return `(value, issue)`,
regular expressions that must match a whole field, a status derived from the issues, and 76
tests in `tests/` that pin every refusal we once got wrong. Most of the effort was not code but
reading: every distinct value of every field, until the parser had an answer for each.

**Learning outcomes.**
- Data wrangling at its core is enumeration: list every value a field takes, then decide what
  each means.
- "Unknown" is a valid output. A pipeline that can say *I cannot read this* is more useful than
  one that always answers.
- A partial parse is worse than none, because it looks like an answer.

> *Each of us: add a line on what you personally took from this stage.*